In [1]:
import pandas as pd
import openpyxl
import numpy as np
import geopandas as gpd
#import fiona
import mercantile #to convert quadkey to geometry
from shapely.geometry import box
import time
import os

import matplotlib.pyplot as plt
import seaborn as sns
import yaml

with open("../config.yaml", "r") as file:
        config = yaml.safe_load(file)

output_dir = "../data/clean/"

config
#import fiona
#fiona.listlayers("ADMIN_EXPRESS.gpkg")


{'input_data': {'file1': '../data/raw/2024-10-01_performance_mobile_tiles.parquet',
  'file2': '../data/raw/ADE_4-0_GPKG_WGS84G_FRA-ED2026-01-19.gpkg',
  'file3': '../data/raw/fichier_diffusion_2024.xlsx',
  'file4': '../data/raw/2024-10-01_performance_fixed_tiles.parquet'},
 'output_data': {'file1': '../data/clean/file1_cleaned.csv',
  'file2': '../data/clean/file2_cleaned.csv'}}

In [2]:
#Read input files
#mobile_perf_df = pd.read_parquet(config['input_data']['file1'])
#mobile_perf_df = internet_perf_df.drop(columns = ['tile', 'tile_x', 'tile_y'])

internet_perf_df = pd.read_parquet(config['input_data']['file4'])
internet_perf_df = internet_perf_df.drop(columns = ['tile', 'tile_x', 'tile_y'])

#read rural zone file
rural_df = pd.read_excel(config['input_data']['file3'], sheet_name = "Maille communale")
rural_df = rural_df.drop(index = range(0,3))
rural_df.columns = rural_df.iloc[0]
rural_df = rural_df.drop(rural_df.index[0]).reset_index(drop=True)
internet_perf_df.head()

,quadkey,avg_d_kbps,avg_u_kbps,avg_lat_ms,avg_lat_down_ms,avg_lat_up_ms,tests,devices
0,0022133222312323,33682,3278,99,208.0,247.0,1,1
1,0022133222330031,97271,14686,192,400.0,386.0,2,2
2,0022133222330033,92047,27325,101,258.0,198.0,1,1
3,0022232311221201,537,255,1001,NaN,3555.0,1,1
4,0022302331120033,104525,4896,100,562.0,170.0,1,1


In [3]:
internet_perf_df.head()

,quadkey,avg_d_kbps,avg_u_kbps,avg_lat_ms,avg_lat_down_ms,avg_lat_up_ms,tests,devices
0,0022133222312323,33682,3278,99,208.0,247.0,1,1
1,0022133222330031,97271,14686,192,400.0,386.0,2,2
2,0022133222330033,92047,27325,101,258.0,198.0,1,1
3,0022232311221201,537,255,1001,NaN,3555.0,1,1
4,0022302331120033,104525,4896,100,562.0,170.0,1,1


In [4]:
#Extract regions, depts and communes from GPKG file
regions = gpd.read_file(config['input_data']['file2'], layer = 'REGION')

depts = gpd.read_file(config['input_data']['file2'], layer = 'DEPARTEMENT')

communes = gpd.read_file(config['input_data']['file2'], layer = 'COMMUNE')


In [5]:
communes.code_insee.unique()

<ArrowStringArray>
['01001', '01002', '01004', '01005', '01006', '01031', '01007', '01008',
 '01009', '01010',
 ...
 '97605', '97608', '97607', '97614', '97602', '97603', '97601', '97606',
 '97609', '97617']
Length: 34877, dtype: str

In [6]:
#convert quadkeys to geometry
def quadkey_to_geom(qk):
    tile = mercantile.quadkey_to_tile(qk)
    bounds = mercantile.bounds(tile)
    
    return box(
        bounds.west,
        bounds.south,
        bounds.east,
        bounds.north
    )

#Apply function of my geopadas df: quakey to geometry and add geometry column
start_time = time.time()
print("Converting quadkeys to geometry in progress...")
internet_perf_df["geometry"] = internet_perf_df["quadkey"].apply(quadkey_to_geom)
end_time = time.time()
print(f"Converting quadkeys to geometry finished. Time taken: {end_time - start_time: .2f}seconds")

Converting quadkeys to geometry in progress...


Converting quadkeys to geometry finished. Time taken:  402.12seconds


In [7]:
#Convert pandas df to Geopandas DataFrame
def convert_df_to_gdf(df):
    """
    Input: pandas df
    convert pandas to geopandas df
    Outpu: geopandas df
    """
    start_time = time.time()
    print("Converting Pandas DF to GeoPandas DF in progress ...")
    df1 = gpd.GeoDataFrame(df, geometry="geometry", crs="EPSG:4326") #EPSG:4326 = WGS84     
    end_time = time.time()
    print(f"Pandas df is converted to geopandas df. Time take is: {end_time - start_time: .2f}seconds")

    return df1

internet_perf_gdf = convert_df_to_gdf(internet_perf_df) #Do not change df name
display(internet_perf_gdf.head())


Converting Pandas DF to GeoPandas DF in progress ...


Pandas df is converted to geopandas df. Time take is:  0.83seconds


,quadkey,avg_d_kbps,avg_u_kbps,avg_lat_ms,avg_lat_down_ms,avg_lat_up_ms,tests,devices,geometry
0,0022133222312323,33682,3278,99,208.0,247.0,1,1,"POLYGON ((-160.01587 70.64177, -160.01587 70.6..."
1,0022133222330031,97271,14686,192,400.0,386.0,2,2,"POLYGON ((-160.02686 70.63631, -160.02686 70.6..."
2,0022133222330033,92047,27325,101,258.0,198.0,1,1,"POLYGON ((-160.02686 70.63448, -160.02686 70.6..."
3,0022232311221201,537,255,1001,NaN,3555.0,1,1,"POLYGON ((-171.85913 66.95373, -171.85913 66.9..."
4,0022302331120033,104525,4896,100,562.0,170.0,1,1,"POLYGON ((-166.09131 68.87144, -166.09131 68.8..."


In [ ]:
from tqdm import tqdm

with tqdm(total=4, desc="Processing Ookla data") as pbar:

    mob_perf_gdf = gpd.read_parquet(file)
    pbar.update(1)

    mob_perf_dpt = gpd.sjoin(
        mob_perf_gdf,
        depts,
        how="left",
        predicate="within"
    )
    pbar.update(1)

    mob_perf_reg = gpd.sjoin(
        mob_perf_dpt,
        regs,
        how="left",
        predicate="within"
    )
    pbar.update(1)

    mob_perf_reg.to_parquet("output.parquet")
    pbar.update(1)

In [8]:
# make sjoin for departments and regions and communes
def get_france_regions_gdf(gdf): 
    """
    Input: geopandas df (performance gdf & admin gdf)
    join geopandas df with region gdf
    Outpu: geopandas df combined for FRance regions
    """
    
    start_time = time.time()
    
    print("Adding adminitrative regions to the data frame is ongoing ...")
    gdf0 = gpd.sjoin(
        gdf,
        regions,
        how = "inner",
        predicate = "intersects"
    )
    gdf0 = gdf0.drop(columns = "index_right")
    
    end_time = time.time()
    print(f"Adding adminitrative regions to the data frame is finished. Time taken is: {end_time - start_time: .2f}seconds")
    
    return gdf0
    
def add_dept_and_comm_to_perf_gdf(gdf1, gdf2):
    """
    Input: geopandas df (performance gdf & admin gdf)
    join geopandas df to geo to geopandas df
    Outpu: geopandas df combined
    """
    
    start_time = time.time()
    
    print("Adding adminitrative classes to the geo data frame is ongoing ...")
    gdf0 = gpd.sjoin(
        gdf1,
        gdf2,
        how = "left",
        predicate = "intersects"
    )
    gdf0 = gdf0.drop(columns = "index_right")
    
    end_time = time.time()
    print(f"Adding adminitrative classes to the geo data frame is finished. Time taken is: {end_time - start_time: .2f}seconds")
    
    return gdf0

def combine_reg_dept_comm(gdf):
    """
    combines the 4 gdf: internet_perf_gdf + regions + depts + communes
    """
    
    internet_perf_reg_fr_gdf =  get_france_regions_gdf(gdf)
    internet_perf_reg_dept_gdf =  add_dept_and_comm_to_perf_gdf(internet_perf_reg_fr_gdf, depts)
    internet_perf_admin_gdf =  add_dept_and_comm_to_perf_gdf(internet_perf_reg_dept_gdf, communes)
    
    return internet_perf_admin_gdf


gdf_perf_admin_all_fr = combine_reg_dept_comm(internet_perf_gdf)
gdf_perf_admin_all_fr.head()

Adding adminitrative regions to the data frame is ongoing ...


Adding adminitrative regions to the data frame is finished. Time taken is:  2357.06seconds
Adding adminitrative classes to the geo data frame is ongoing ...


Adding adminitrative classes to the geo data frame is finished. Time taken is:  380.83seconds
Adding adminitrative classes to the geo data frame is ongoing ...


Adding adminitrative classes to the geo data frame is finished. Time taken is:  25.22seconds


,quadkey,avg_d_kbps,avg_u_kbps,avg_lat_ms,avg_lat_down_ms,avg_lat_up_ms,tests,devices,geometry,cleabs_left,...,date_du_recensement,organisme_recenseur,code_insee_du_canton,code_insee_de_l_arrondissement,code_insee_du_departement,code_insee_de_la_region_right,codes_siren_des_epci,code_siren,code_postal,superficie_cadastrale
1538494,0313133210211310,325781,330396,14,63.0,1496.0,1,1,"POLYGON ((-1.93909 49.70672, -1.93909 49.71027...",REGION__0000000000000028,...,2023-01-01,INSEE,5014,502,50,28,200067205,200065415,50440,14870
1538495,0313133210211333,318053,462155,9,82.0,177.0,1,1,"POLYGON ((-1.93359 49.69606, -1.93359 49.69961...",REGION__0000000000000028,...,2023-01-01,INSEE,5014,502,50,28,200067205,200065415,50440,14870
1538497,0313133210300032,140713,86536,14,167.0,178.0,2,1,"POLYGON ((-1.91711 49.71027, -1.91711 49.71382...",REGION__0000000000000028,...,2023-01-01,INSEE,5014,502,50,28,200067205,200065415,50440,14870
1538498,0313133210300033,395560,291002,17,337.0,480.0,6,1,"POLYGON ((-1.91162 49.71027, -1.91162 49.71382...",REGION__0000000000000028,...,2023-01-01,INSEE,5014,502,50,28,200067205,200065415,50440,14870
1538499,0313133210300132,237308,257188,10,337.0,145.0,2,1,"POLYGON ((-1.89514 49.71027, -1.89514 49.71382...",REGION__0000000000000028,...,2023-01-01,INSEE,5014,502,50,28,200067205,200065415,50440,14870


In [10]:
#gdf_perf_admin_all.code_siren.unique()
#gdf_perf_admin_all.columns
gdf_perf_admin_all_fr.isna().sum()
gdf_perf_admin_all_fr.shape

(230321, 36)

In [25]:
# Renaming and selecting columns

def column_clean_up(gdf):
    """
    input = 
    Keep only france regions+depts+communes
    Rename the columns
    outpu: france_gdf
    """
    #gdf = combine_reg_dept_comm()
    selected_col = ['avg_d_kbps', 'avg_u_kbps', 'avg_lat_ms', 'avg_lat_down_ms',
               'avg_lat_up_ms', 'tests', 'devices','nom_officiel_en_majuscules_left',
               'code_insee_left', 'code_siren_left','nom_officiel_en_majuscules_right',
                'code_insee_right', 'nom_officiel_en_majuscules', 'statut',
               'code_insee', 'population', 'date_du_recensement',
               'code_postal', 'superficie_cadastrale'] #'quadkey',  'geometry': I removed the geodata

    new_col_names = ['avg_down_kbps', 'avg_up_kbps', 'avg_lat_ms', 'avg_lat_down_ms',
                'avg_lat_up_ms', 'nbr_tests', 'nbr_devices','region_name',
               'code_insee_region', 'code_siren_region','department_name',
                'code_insee_dept', 'commune_name', 'commune_status',
                'code_insee_comm', 'comm_population', 'date_du_recensement',
                'zip_code', 'superficie_cadastrale'
                ]

    gdf = gdf[selected_col] #reduce number of columns
    gdf.columns = new_col_names #rename columns
    gdf[['avg_down_kbps', 'avg_up_kbps']] = (gdf[['avg_down_kbps', 'avg_up_kbps']]/1000).round(2)
    gdf = gdf.rename(columns = {'avg_down_kbps':'avg_down_mbps' , 'avg_up_kbps':'avg_up_mbps'})
    return gdf
    

#gdf_perf_admin_all = combine_reg_dept_comm() #gdf with all countries

df_perf_admin_fr = column_clean_up(gdf_perf_admin_all_fr) #gdf france
#gdf_perf_admin_fr.to_csv(f"{output_dir}gdf_perf_admin_fr_inner.csv", index=False, encoding= "utf-8", sep = ";")


In [26]:
df_perf_admin_fr.info()
df_perf_admin_fr.isna().sum()


<class 'pandas.DataFrame'>
Index: 230321 entries, 1538494 to 6271875
Data columns (total 19 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   avg_down_mbps          230321 non-null  float64       
 1   avg_up_mbps            230321 non-null  float64       
 2   avg_lat_ms             230321 non-null  int64         
 3   avg_lat_down_ms        229336 non-null  float64       
 4   avg_lat_up_ms          229357 non-null  float64       
 5   nbr_tests              230321 non-null  int64         
 6   nbr_devices            230321 non-null  int64         
 7   region_name            230321 non-null  str           
 8   code_insee_region      230321 non-null  str           
 9   code_siren_region      230321 non-null  str           
 10  department_name        230321 non-null  str           
 11  code_insee_dept        230321 non-null  str           
 12  commune_name           230321 non-null  str          

avg_down_mbps               0
avg_up_mbps                 0
avg_lat_ms                  0
avg_lat_down_ms           985
avg_lat_up_ms             964
nbr_tests                   0
nbr_devices                 0
region_name                 0
code_insee_region           0
code_siren_region           0
department_name             0
code_insee_dept             0
commune_name                0
commune_status              0
code_insee_comm             0
comm_population             0
date_du_recensement         0
zip_code                 1812
superficie_cadastrale       0
dtype: int64

In [27]:
df_perf_admin_fr.commune_name.unique()

<ArrowStringArray>
[               'LA HAGUE',   'CHERBOURG-EN-COTENTIN',
             'NOUAINVILLE',              'MARTINVAST',
             'FERMANVILLE',       'MAUPERTUS-SUR-MER',
                'THEVILLE',              'CARNEVILLE',
            'VICQ-SUR-MER',              'DIGOSVILLE',
 ...
               'SAINT-LEU',             'LES AVIRONS',
            'L'ETANG-SALE',                  'CILAOS',
               'LE TAMPON',              'ENTRE-DEUX',
              'BRAS-PANON', 'LA PLAINE-DES-PALMISTES',
              'PETITE-ILE',          'SAINT-PHILIPPE']
Length: 27371, dtype: str

In [37]:
# Convert the downmoad and upload speed fro kbps to mbps
def merge_with_rural_zone(gdf):
    #convert gdf to df
    gdf1 = gdf.copy()
    gdf1 = pd.DataFrame(gdf)
    gdf1 = pd.merge(gdf1, rural_df[["CODGEO","LIBDENS"]] , how = "left", left_on = "code_insee_comm", right_on = "CODGEO")
    gdf1 = gdf1.drop(columns = ["CODGEO","date_du_recensement"])
    return gdf1
internet_perf_df_fr= merge_with_rural_zone(df_perf_admin_fr)
internet_perf_df_fr.to_csv(f"{output_dir}perf_admin_fr_df.csv", index=False, encoding= "utf-8", sep = ",")
internet_perf_df_fr.head()


,avg_down_mbps,avg_up_mbps,avg_lat_ms,avg_lat_down_ms,avg_lat_up_ms,nbr_tests,nbr_devices,region_name,code_insee_region,code_siren_region,department_name,code_insee_dept,commune_name,commune_status,code_insee_comm,comm_population,zip_code,superficie_cadastrale,LIBDENS
0,325.78,330.40,14,63.0,1496.0,1,1,NORMANDIE,28,200053403,MANCHE,50,LA HAGUE,Commune simple,50041,11484,50440,14870,Rural
1,318.05,462.16,9,82.0,177.0,1,1,NORMANDIE,28,200053403,MANCHE,50,LA HAGUE,Commune simple,50041,11484,50440,14870,Rural
2,140.71,86.54,14,167.0,178.0,2,1,NORMANDIE,28,200053403,MANCHE,50,LA HAGUE,Commune simple,50041,11484,50440,14870,Rural
3,395.56,291.00,17,337.0,480.0,6,1,NORMANDIE,28,200053403,MANCHE,50,LA HAGUE,Commune simple,50041,11484,50440,14870,Rural
4,237.31,257.19,10,337.0,145.0,2,1,NORMANDIE,28,200053403,MANCHE,50,LA HAGUE,Commune simple,50041,11484,50440,14870,Rural


In [34]:
display(internet_perf_df_fr.describe())
display(internet_perf_df_fr[internet_perf_df_fr.commune_name == "LA HAGUE"])

,avg_down_mbps,avg_up_mbps,avg_lat_ms,avg_lat_down_ms,avg_lat_up_ms,nbr_tests,nbr_devices,comm_population,superficie_cadastrale
count,230321.000000,230321.000000,230321.000000,229336.000000,229357.000000,230321.000000,230321.000000,2.303210e+05,2.303210e+05
mean,317.818701,232.911026,19.811485,380.229327,345.934905,7.595339,2.501522,2.592274e+04,3.025710e+03
std,254.882592,186.134935,49.912854,593.828524,581.494782,17.484089,4.118919,1.398961e+05,1.585033e+04
min,0.000000,0.000000,0.000000,4.000000,4.000000,1.000000,1.000000,3.000000e+00,0.000000e+00
25%,121.130000,76.810000,7.000000,105.000000,60.000000,1.000000,1.000000,9.720000e+02,8.800000e+02
50%,286.820000,216.240000,11.000000,201.000000,150.000000,3.000000,1.000000,3.084000e+03,1.600000e+03
75%,442.000000,339.620000,18.000000,403.000000,375.000000,7.000000,2.000000,1.049300e+04,3.140000e+03
max,4615.530000,3457.220000,1521.000000,9960.000000,9974.000000,1493.000000,147.000000,2.103778e+06,1.836000e+06


,avg_down_mbps,avg_up_mbps,avg_lat_ms,avg_lat_down_ms,avg_lat_up_ms,nbr_tests,nbr_devices,region_name,code_insee_region,code_siren_region,department_name,code_insee_dept,commune_name,commune_status,code_insee_comm,comm_population,zip_code,superficie_cadastrale,LIBDENS
0,325.78,330.40,14,63.0,1496.0,1,1,NORMANDIE,28,200053403,MANCHE,50,LA HAGUE,Commune simple,50041,11484,50440,14870,Rural
1,318.05,462.16,9,82.0,177.0,1,1,NORMANDIE,28,200053403,MANCHE,50,LA HAGUE,Commune simple,50041,11484,50440,14870,Rural
2,140.71,86.54,14,167.0,178.0,2,1,NORMANDIE,28,200053403,MANCHE,50,LA HAGUE,Commune simple,50041,11484,50440,14870,Rural
3,395.56,291.00,17,337.0,480.0,6,1,NORMANDIE,28,200053403,MANCHE,50,LA HAGUE,Commune simple,50041,11484,50440,14870,Rural
4,237.31,257.19,10,337.0,145.0,2,1,NORMANDIE,28,200053403,MANCHE,50,LA HAGUE,Commune simple,50041,11484,50440,14870,Rural
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
335,13.23,0.32,38,1094.0,6570.0,1,1,NORMANDIE,28,200053403,MANCHE,50,LA HAGUE,Commune simple,50041,11484,50440,14870,Rural
337,348.65,145.40,11,77.0,72.0,1,1,NORMANDIE,28,200053403,MANCHE,50,LA HAGUE,Commune simple,50041,11484,50440,14870,Rural
339,49.03,45.53,14,389.0,223.0,1,1,NORMANDIE,28,200053403,MANCHE,50,LA HAGUE,Commune simple,50041,11484,50440,14870,Rural
341,667.16,403.42,7,44.0,99.0,1,1,NORMANDIE,28,200053403,MANCHE,50,LA HAGUE,Commune simple,50041,11484,50440,14870,Rural
